# Обучение BPE и первые примеры токенизации

Загружаем подготовленные train/val из Parquet. Обучаем один общий byte-level BPE на семпле из 500 000 примеров train'а. Используем `code_without_comments` как колонку с кодом, это будет нашим baseline. Затем смотрим на словарь: распределение токенов, пример токинезации и распределедения длин.

In [43]:
import sys
from pathlib import Path

project_dir = Path.cwd()
if not (project_dir / "lib").is_dir():
    project_dir = project_dir.parent
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from lib.utils.paths import PROJECT_DIR

from collections import Counter
from html import escape
from random import Random
from textwrap import dedent

import pandas as pd
from datasets import Dataset, DatasetDict
from IPython.display import HTML, display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer
from tqdm.auto import tqdm

from tokenizers import Tokenizer
from lib.data_processing.tokenizer import train_tokenizer, tokenize_dataset

In [ ]:
SEED = 62
SAMPLE_SIZE = 500_000
VOCAB_SIZE = 16_384
MIN_FREQUENCY = 2
BATCH_SIZE = 10_000
CODE_COLUMN = "code_without_comments"
FREQUENCY_SAMPLE_SIZE = 50_000

PREPARED_DIR = PROJECT_DIR / "data" / "prepared"
CACHE_DIR = PROJECT_DIR / "data" / ".cache" / "tokenization"
TOKENIZER_PATH = PROJECT_DIR / "data" / "tokenizer.json"

## Загрузка подготовленных данных

Читаем train и val отдельно: в val дополнительно сохранена колонка `answer`.
Hugging Face Datasets использует дисковый Arrow-кеш; целиком переводить данные в pandas не будем.

In [30]:
train = Dataset.from_parquet(
    str(PREPARED_DIR / "train.parquet"), cache_dir=str(CACHE_DIR),
)
val = Dataset.from_parquet(
    str(PREPARED_DIR / "val.parquet"), cache_dir=str(CACHE_DIR),
)

pd.DataFrame([
    {"split": "train", "rows": len(train), "columns": train.column_names},
    {"split": "val", "rows": len(val), "columns": val.column_names},
])

,split,rows,columns
0,train,11510485,"[question, code, code_without_comments]"
1,val,7954,"[question, code, code_without_comments, answer]"


## Выборка для обучения BPE

Посторим семпл выбоку для обучения. В обучение BPE попадают только тексты из train.

In [31]:
sample_indices = Random(SEED).sample(range(len(train)), SAMPLE_SIZE)
train_sample = train.select(sorted(sample_indices)).select_columns([
    "question", CODE_COLUMN,
])
print(f"Выбрано примеров: {len(train_sample):,}")

Выбрано примеров: 500,000


## Обучение общего токенайзера

Вопрос и решение передаются отдельными текстами в общий корпус. ByteLevel сохраняет
пробелы, переносы и регистр. После обучения токенайзер автоматически добавляет
`<bos>` и `<eos>` при кодировании.

In [32]:
tokenizer = train_tokenizer(
    train_sample,
    code_column=CODE_COLUMN,
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    batch_size=BATCH_SIZE,
)

tokenizer.save(str(TOKENIZER_PATH))
print(f"Фактический размер словаря: {tokenizer.get_vocab_size():,}")
print(f"Токенайзер сохранён: {TOKENIZER_PATH}")




Фактический размер словаря: 16,384
Токенайзер сохранён: /Users/maks62wip/ML/NLP/HW-2/data/tokenizer.json


## Изучение словаря

Посмотрим, что у нас лежит в токенайзере: начало словаря и случайные токены.

In [33]:
vocab_df = pd.DataFrame(
    tokenizer.get_vocab().items(), columns=["token", "id"],
).sort_values("id").reset_index(drop=True)
vocab_df["decoded"] = vocab_df["id"].map(
    lambda token_id: repr(tokenizer.decode([token_id], skip_special_tokens=False))
)
vocab_df = vocab_df[["id", "token", "decoded"]]

display(vocab_df.head(12))
display(vocab_df.sample(20, random_state=SEED).sort_values("id"))

,id,token,decoded
0,0,<pad>,'<pad>'
1,1,<bos>,'<bos>'
2,2,<eos>,'<eos>'
3,3,<unk>,'<unk>'
4,4,!,'!'
5,5,"""","'""'"
6,6,#,'#'
7,7,$,'$'
8,8,%,'%'
9,9,&,'&'


,id,token,decoded
484,484,aining,'aining'
578,578,ess,'ess'
901,901,Ġbut,' but'
1391,1391,abella,'abella'
2531,2531,800,'800'
3734,3734,Ġpace,' pace'
6877,6877,Ġdig,' dig'
7141,7141,Ġlitter,' litter'
10311,10311,Ġorn,' orn'
10550,10550,Ġparties,' parties'


### Частые токены

Посчитаем количество вхождений каждого токена в данные и посмотрим на самые популярные.

In [34]:
question_counts = Counter()
code_counts = Counter()

for batch in tqdm(
    train_sample.iter(batch_size=BATCH_SIZE),
    total=(len(train_sample) + BATCH_SIZE - 1) // BATCH_SIZE,
    desc="Считаем частоты токенов",
):
    for encoding in tokenizer.encode_batch(batch["question"], add_special_tokens=False):
        question_counts.update(encoding.ids)
    for encoding in tokenizer.encode_batch(batch[CODE_COLUMN], add_special_tokens=False):
        code_counts.update(encoding.ids)

Считаем частоты токенов:   0%|          | 0/50 [00:00<?, ?it/s]

In [35]:
frequency_df = vocab_df.copy()
frequency_df["question_count"] = frequency_df["id"].map(lambda token_id: question_counts[token_id])
frequency_df["code_count"] = frequency_df["id"].map(lambda token_id: code_counts[token_id])
frequency_df["total_count"] = frequency_df["question_count"] + frequency_df["code_count"]

display(frequency_df.nlargest(25, "total_count").reset_index(drop=True))

,id,token,decoded,question_count,code_count,total_count
0,66,_,'_',347,5645085,5645432
1,262,ĊĠĠĠ,'\n ',0,3159207,3159207
2,265,Ġ=,' =',1320,2698024,2699344
3,17,.,'.',1098060,168789,1266849
4,287,Ġresult,' result',292,1001793,1002085
5,298,Ġtotal,' total',115221,720271,835492
6,15,",","','",705075,25423,730498
7,296,Ġthe,' the',700602,671,701273
8,306,Ġof,' of',641600,653,642253
9,317,Ġ*,' *',77,554891,554968


У нас есть токены встречающиеся и в коде и в тексте часто. Но в среднем кодоспецефичных токенов больше.

### Случайные примеры

Посмотрим на рандомные закодированные примеры, возьмем по 1 штуке из каждой выборке и затокинезируем их.

In [36]:
def show_tokenized_example(split_name, row_index, example):
    display(HTML(f"<h3>{split_name} · строка {row_index:,}</h3>"))
    question = escape(example["question"].strip())
    code = dedent(example[CODE_COLUMN]).strip("\n")
    display(HTML(f"""
        <div style="
            box-sizing: border-box; max-width: 100%;
            padding: 18px 22px; margin: 8px 0;
            border: 1px solid #cbd5e1; border-radius: 12px;
            background: #f8fafc; color: #0f172a;
        ">
            <div style="font-weight: 600; color: #64748b; margin-bottom: 10px;">
                Вопрос:
            </div>
            <div style="font-size: 16px; line-height: 1.6; overflow-wrap: anywhere;">
                {question}
            </div>
        </div>
        <h4 style="margin: 18px 0 8px;">Решение:</h4>
    """))
    formatter = HtmlFormatter(
        noclasses=True,
        cssstyles="color: #111827; padding: 14px; border-radius: 10px; overflow-x: auto;",
        prestyles="color: inherit; background: transparent; margin: 0;",
    )
    display(HTML(highlight(code, PythonLexer(), formatter)))
    if "answer" in example:
        print("Сохранённый answer:", example["answer"])

    fields = [("question", "input_ids"), (CODE_COLUMN, "target_ids")]
    display(pd.DataFrame([
        {
            "field": text_column,
            "characters": len(example[text_column]),
            "tokens": len(example[ids_column]),
            "round_trip": tokenizer.decode(example[ids_column]) == example[text_column],
        }
        for text_column, ids_column in fields
    ]))

    for text_column, ids_column in fields:
        display(HTML(f"<h4>Токены: {escape(text_column)}</h4>"))
        ids = example[ids_column]
        tokens_df = pd.DataFrame({
            "position": range(len(ids)),
            "id": ids,
            "token": [tokenizer.id_to_token(token_id) for token_id in ids],
            "decoded": [
                repr(tokenizer.decode([token_id], skip_special_tokens=False))
                for token_id in ids
            ],
        })
        display(tokens_df.head(40))

In [37]:
example_rng = Random(SEED)
tokenized_examples = {}

for split_name, dataset in [("train", train), ("val", val)]:
    row_index = example_rng.randrange(len(dataset))
    tokenized = tokenize_dataset(
        dataset.select([row_index]), tokenizer,
        code_column=CODE_COLUMN, batch_size=BATCH_SIZE,
    )
    example = tokenized[0]
    tokenized_examples[split_name] = example
    show_tokenized_example(split_name, row_index, example)

Tokenizing questions and code:   0%|          | 0/1 [00:00<?, ? examples/s]

,field,characters,tokens,round_trip
0,question,102,26,True
1,code_without_comments,122,36,True


,position,id,token,decoded
0,0,1,<bos>,'<bos>'
1,1,4484,Jason,'Jason'
2,2,380,Ġhas,' has'
3,3,337,Ġ3,' 3'
4,4,571,Ġmore,' more'
5,5,579,Ġthan,' than'
6,6,767,Ġhalf,' half'
7,7,437,Ġas,' as'
8,8,352,Ġmany,' many'
9,9,536,Ġbooks,' books'


,position,id,token,decoded
0,0,1,<bos>,'<bos>'
1,1,328,def,'def'
2,2,326,Ġsolve,' solve'
3,3,329,(),'()'
4,4,330,Ġ->,' ->'
5,5,321,Ġint,' int'
6,6,29,:,':'
7,7,262,ĊĠĠĠ,'\n '
8,8,536,Ġbooks,' books'
9,9,66,_,'_'


Tokenizing questions and code:   0%|          | 0/1 [00:00<?, ? examples/s]

Сохранённый answer: 200.0


,field,characters,tokens,round_trip
0,question,430,92,True
1,code_without_comments,295,78,True


,position,id,token,decoded
0,0,1,<bos>,'<bos>'
1,1,2296,Michael,'Michael'
2,2,2700,Ġloves,' loves'
3,3,2071,Ġpainting,' painting'
4,4,332,Ġand,' and'
5,5,8486,Ġdraws,' draws'
6,6,4783,Ġportraits,' portraits'
7,7,306,Ġof,' of'
8,8,516,Ġhis,' his'
9,9,341,Ġl,' l'


,position,id,token,decoded
0,0,1,<bos>,'<bos>'
1,1,328,def,'def'
2,2,326,Ġsolve,' solve'
3,3,329,(),'()'
4,4,330,Ġ->,' ->'
5,5,321,Ġint,' int'
6,6,29,:,':'
7,7,262,ĊĠĠĠ,'\n '
8,8,6485,Ġwife,' wife'
9,9,66,_,'_'


Посмотрим на длины примеров в токенах:

In [38]:
train_sample_tokenized = tokenize_dataset(
    train_sample, tokenizer,
    code_column=CODE_COLUMN,
    batch_size=BATCH_SIZE,
)

val_tokenized = tokenize_dataset(
    val, tokenizer,
    code_column=CODE_COLUMN,
    batch_size=BATCH_SIZE,
)

for split_name, dataset in [
    ("train_sample", train_sample_tokenized),
    ("val", val_tokenized),
]:
    display(HTML(f"<h3>{split_name}:</h3>"))

    lengths = dataset.select_columns([
        "input_length", "target_length",
    ]).to_pandas()

    display(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

Tokenizing questions and code:   0%|          | 0/500000 [00:00<?, ? examples/s]

Tokenizing questions and code:   0%|          | 0/7954 [00:00<?, ? examples/s]

,input_length,target_length
count,500000.000000,500000.000000
mean,46.064874,67.370128
std,17.080117,29.193428
min,6.000000,16.000000
50%,43.000000,60.000000
90%,69.000000,105.000000
95%,78.000000,124.000000
99%,100.000000,167.000000
max,270.000000,410.000000


,input_length,target_length
count,7954.000000,7954.000000
mean,46.216118,67.877923
std,17.352433,29.149723
min,11.000000,16.000000
50%,43.000000,61.000000
90%,70.000000,106.000000
95%,78.000000,125.000000
99%,102.470000,163.470000
max,168.000000,341.000000


Я бы пока не фильтровал выборку по длине: длины умеренные, а train_sample и val очень близки.

## Обучение итогового токенайзера поверх всего train

In [39]:
TOKENIZER_PATH = PROJECT_DIR / "data" / "tokenizer_full_train.json"

print(f"Обучаем BPE на {len(train):,} примерах train")

tokenizer = train_tokenizer(
    train,
    code_column=CODE_COLUMN,
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    batch_size=BATCH_SIZE,
)

tokenizer.save(str(TOKENIZER_PATH))

print(f"Размер словаря: {tokenizer.get_vocab_size():,}")
print(f"Сохранён: {TOKENIZER_PATH}")

Обучаем BPE на 11,510,485 примерах train



Размер словаря: 16,384
Сохранён: /Users/maks62wip/ML/NLP/HW-2/data/tokenizer_full_train.json


In [42]:
final_tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

Теперь токинезуем все и сохраним:

In [45]:
token_columns = ["input_ids", "target_ids", "input_length", "target_length"]

train_tokenized = tokenize_dataset(
    train.select_columns(["question", CODE_COLUMN]),
    final_tokenizer,
    code_column=CODE_COLUMN,
    batch_size=BATCH_SIZE,
).select_columns(token_columns)

val_tokenized = tokenize_dataset(
    val,
    final_tokenizer,
    code_column=CODE_COLUMN,
    batch_size=BATCH_SIZE,
).select_columns(["answer", *token_columns])

tokenized_data = DatasetDict({
    "train": train_tokenized,
    "val": val_tokenized,
})

Tokenizing questions and code:   0%|          | 0/11510485 [00:00<?, ? examples/s]

Tokenizing questions and code:   0%|          | 0/7954 [00:00<?, ? examples/s]

In [46]:
output_dir = PROJECT_DIR / "data" / "tokenized"
tokenized_data.save_to_disk(str(output_dir), max_shard_size="1GB")
final_tokenizer.save(str(output_dir / "tokenizer.json"))  # на всякий, чтобы лежал рядом

Saving the dataset (0/9 shards):   0%|          | 0/11510485 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7954 [00:00<?, ? examples/s]

Теперь можно учиться поверх этого.